In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import matplotlib.pyplot as plt
from _split_format import split_and_format_data
from _xgboost_model import train_xgboost_model_random

import json
from datetime import datetime

from sklearn.metrics import(
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    accuracy_score,
    fbeta_score,
    roc_auc_score,
    matthews_corrcoef,
    precision_recall_curve,
    roc_curve,
    auc,
)

In [ ]:
SEX = "Male"
AGE_GROUP = "50-54"
DATA_PATH = f"/data/workdata/709656/Anne/Data_files/cohort_data/cohort_{AGE_GROUP.replace('-','_to_')}.parquet"
RANDOM_STATE=42
MODEL_NAME="emily_20260722"
MAXIMIZE_METRIC = "f2" # or pr_auc, recall, precision, roc_auc
#MIN_PRECISION =0.15

COHORT_NAME= f"{SEX.lower()}_{AGE_GROUP}"
OUTPUT_DIR = Path(
    f"../XGBoost_results/{COHORT_NAME}"
)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
# Load, split, preprocess
(
X_train_raw,
X_cal_raw,
X_test_raw,
y_train,
y_cal,
y_test,
id_fit,
if_cal,
id_test,
preprocessor
) = split_and_format_data(
    data_path=DATA_PATH,
    drop_cols=[
        "pnr",
        "family_id",
        "in_dk",
        "de_age",
        "alive",
        "de_parish",
        "de_region",
        "de_municipality",
        "de_time_to_death",
        "de_age_at_death",
        "se_educ_date",
        "de_sex",
    ],
sex_filter = [SEX],
target_col="early_death",
test_size=0.3,
cal_size_within_train=0.3,
random_state=RANDOM_STATE,
stratify_on_year=True,
year_col="year",
)
print("Train shape:", X_train_raw.shape)
print("Cal shape:", X_cal_raw.shape)
print("Test shape:", X_test_raw.shape)
print("Train death rate:", y_train.mean())
print("Cal death rate:", y_cal.mean())
print("Test death rate:", y_test.mean())

In [ ]:
# death rates
raw_train_death_rate = float(np.mean(y_train))
cal_death_rate = float(np.mean(y_cal))
test_death_rate =float(np.mean(y_test))

In [ ]:
#Scaleposweight
spw= (1-y_train.mean()) /y_train.mean()
print(spw)

In [ ]:
#Hyperparameters
param_grid = {
    "max_depth": [2,3,4],
    "learning_rate": [0.01,0.03,0.05],
    "n_estimators": [400,800,1000],
    "subsample": [0.5, 0.6, 0.7],
    "colsample_bytree": [0.3, 0.5, 0.7],
    "gamma":[2,5,10,20],
    "min_child_weight":[10,20,40],
    "reg_lambda":[20,40,80],
    "reg_alpha":[1,5,10],
}

search_seed = abs(hash(COHORT_NAME)) % (2**32)
best_model, best_model_params, model_thr = train_xgboost_model_random(
    X_train_raw,
    y_train,
    preprocessor=preprocessor,
    param_grid=param_grid,
    cv_folds=3,
    random_state=search_seed,
    maximize="f2",
    #min_precision=0.15,
    scale_pos_weight=spw,
    n_iter=60,
)

#Save model as json
best_model.named_steps["model"].save_model(
    OUTPUT_DIR / f"best_model_{COHORT_NAME}.json")

## Calibration

In [ ]:
calibrated_model = CalibratedClassifierCV(
    estimator=best_model,
    method="isotonic",
    cv= "prefit"
)
calibrated_model.fit(X_cal_raw,y_cal)

In [ ]:
## Predicted probability for survivors vs deaths + gap. Calibrated model.
# split data by outcome
X_survivors = X_test_raw[y_test == 0].copy()
X_deaths = X_test_raw[y_test == 1].copy()
#### OBS model
    #  gap
p_survivors = calibrated_model.predict_proba(X_survivors)[:, 1].mean()
p_deaths = calibrated_model.predict_proba(X_deaths)[:, 1].mean()
total_gap = p_deaths - p_survivors

print("=== Predicted probabilites on test Calibrated model ===")
print(f"Survivors (y=0), mean p(death): {p_survivors:.4f}")
print(f"Early deaths (y=1), mean p(death): {p_deaths:.4f}")
print(f"Mortality gap (death-survivors): {total_gap:.4f}")

In [ ]:
# Bootstrap 95% CI for the calibrated mortality gap (test set)
def bootstrap_pred_probs(y_true, y_prob, n_bootstrap=2000, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    p_surv, p_death = y_prob[y_true == 0], y_prob[y_true == 1]

    obs = {
        "survivors": p_surv.mean(),
        "decedents": p_death.mean(),
        "gap":       p_death.mean() - p_surv.mean(),
    }

    boot = {k: np.empty(n_bootstrap) for k in ("survivors","decedents","gap")}
    for i in range(n_bootstrap):
        s = rng.choice(p_surv,  p_surv.size,  replace=True).mean()
        d = rng.choice(p_death, p_death.size, replace=True).mean()
        boot["survivors"][i] = s
        boot["decedents"][i] = d
        boot["gap"][i]       = d - s

    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in boot.items()}
    return obs, ci

y_prob_cal_test = calibrated_model.predict_proba(X_test_raw)[:, 1]
obs, ci = bootstrap_pred_probs(y_test, y_prob_cal_test)
for k in ("survivors","decedents","gap"):
    print(f"{k:10s} = {obs[k]:.4f}  95% CI [{ci[k][0]:.4f}, {ci[k][1]:.4f}]")

In [ ]:
# Calibration curve
# Raw model
y_prob_raw = best_model.predict_proba(X_test_raw)[:,1]
# Calibrated model
y_prob_cal = calibrated_model.predict_proba(X_test_raw)[:,1]
prob_true_raw, prob_pred_raw = calibration_curve(y_test,y_prob_raw, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test,y_prob_cal, n_bins=10)

plt.figure(figsize=(6,6))

plt.plot(prob_pred_raw, prob_true_raw, marker="o", label="Raw model")
plt.plot(prob_pred_cal, prob_true_cal, marker="o", label="Calibrated model")
plt.plot([0,1],[0,1], linestyle="--", color="gray", label="Perfect calibration")

plt.xlabel("Predicted probability")
plt.ylabel("Observed probability")
plt.title("Calibration Curve")

plt.legend()
plt.savefig( OUTPUT_DIR/ f"calibration_curve_{COHORT_NAME}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Save calibrated model
# joblib.dump(best_model,
# OUTPUT_DIR / f"raw_pipeline_{COHORT_NAME}.joblib")

joblib.dump(calibrated_model,
OUTPUT_DIR / f"calibrated_model_{COHORT_NAME}.joblib")

# X_test_raw.to_parquet(
#     OUTPUT_DIR/ f"X_test_raw_{COHORT_NAME}.parquet", index=False
# )

X_cal_raw.to_parquet(
    OUTPUT_DIR/ f"X_cal_raw_{COHORT_NAME}.parquet", index=False
)
pd.DataFrame({"y_cal": y_cal}).to_parquet(
    OUTPUT_DIR / f"y_cal_raw_{COHORT_NAME}.parquet",
    index=False
)

In [ ]:
#Find optimal calibration threshold
y_prob_cal = calibrated_model.predict_proba(X_cal_raw)[:, 1]

precision_cal, recall_cal, thresholds_cal = precision_recall_curve(y_cal,y_prob_cal)

# Choose threshold that maximizes f2 on calibtaion set
beta=2
p_cal= precision_cal[:-1]
r_cal= recall_cal[:-1]
f2_cal = (1 + beta**2) * (p_cal *r_cal) / (beta**2 * p_cal + r_cal + 1e-12)
best_idx_cal = np.argmax(f2_cal)
calibrated_threshold = float(thresholds_cal[best_idx_cal])

print("Chosen threshold (max f2):", calibrated_threshold)
print("Precision / Recall at chosen thr:", p_cal[best_idx_cal], r_cal[best_idx_cal])
print("F2 at chosen thr:", f2_cal[best_idx_cal])

In [ ]:
# Save calibrated prediction
y_prob_cal = calibrated_model.predict_proba(X_test_raw)[:,1]
y_pred_cal = (y_prob_cal >= calibrated_threshold).astype(int)
pred_cal_df = pd.DataFrame({
    "pnr": id_test.astype(str),
    "y_test": np.asarray(y_test).astype(int),
    "y_proba": y_prob_cal,
    "y_pred": y_pred_cal,
    "calibrated_threshold": calibrated_threshold,
    "test_death_rate": y_test.mean()
})
pred_cal_df.to_parquet(OUTPUT_DIR/ "calibrated_predictions.parquet", index=False)

In [ ]:
# Save calibration data after preprocessing
preprocessor_fitted = best_model.named_steps["preprocess"]
X_cal_processed = preprocessor_fitted.transform(X_cal_raw)

if hasattr(X_cal_processed, "toarray"):
    X_cal_processed = X_cal_processed.toarray()

X_cal_processed = pd.DataFrame(
    X_cal_processed,
    columns=preprocessor_fitted.get_feature_names_out()
)

X_cal_processed.columns = (
    X_cal_processed.columns
    .str.replace("^remainder__", "", regex=True)
    .str.replace("^cat__", "", regex=True)
)

X_cal_processed.to_csv(OUTPUT_DIR / f"X_cal_{COHORT_NAME}.csv", index=False)
pd.DataFrame({"y_cal": y_cal}).to_csv(OUTPUT_DIR /f"y_cal_{COHORT_NAME}.csv", index=False)

In [ ]:
# Calibrated model evaluation on test set
y_prob = calibrated_model.predict_proba(X_test_raw)[:,1]
y_pred = (y_prob >= calibrated_threshold).astype(int)

f1= f1_score(y_test, y_pred)
f2 = fbeta_score(y_test, y_pred, beta=2)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)
bal = balanced_accuracy_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("\n ==== Calibration metric=====" )
print("F1 score:", f1)
print("F2 score:", f2)
print("Precision:", prec)
print("Recall:", rec)
print("ROC-AUC:", roc)
print("PR-AUC (avg prec):", pr)
print("Balanced accuracy:", bal)
print("Accuracy:", acc)
print("MCC:", mcc)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print("\n === Confusion matrix (test set) === ")
print(cm)
print(f"TN:{tn}, FP: {fp}, FN: {fn}, TP:{tp}")
print("Specificity (TNR):", specificity)

In [ ]:
# Save results
time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

results = {
    "model_name": MODEL_NAME,
    "objective":{
        "cohort": f"{COHORT_NAME}",
        "maximize":"f2",
        "eval_metric": "aucpr",
        "min_precision": 0,
        "cv_folds":3,
        "n_iter":60,
        "chosen threshold":float(model_thr),
        "scale_pos_weight":float(spw),
        "threshold_source": "OOF_train",
        "comments": "Class imbalanced handled with scale_pos_weight instead of rebalancing"
    },

    "best_params": best_model_params,

    "death_rate_summary" :{
        "raw_train_death_rate": raw_train_death_rate,
        "test_death_rate": test_death_rate,
        "calibration_death_rate": cal_death_rate,
        #"resampled_train_death_rate": resampled_train_death_rate,
    },

    "metrics": {
        "f1": float(f1),
        "f2": float(f2),
        "precision": float(prec),
        "recall":float(rec),
        "roc_auc":float(roc),
        "pr_auc": float(pr),
        "specificity (TNR)": float(specificity),
        "balanced_accuracy": float(bal),
        "accuracy":float(acc),
        "MCC":float(mcc),
        "threshold_used": float(calibrated_threshold),
    },

    "confusion_matrix_test":{
        "TN":int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP":int(tp),
    },

    "predicted_probabilities_test":{
        "mean_survived":float(np.mean(y_prob[y_test==0])),
        "mean_died":float(np.mean(y_prob[y_test==1])),
        "gap_died_minus_survived": float(np.mean(y_prob[y_test==1])-np.mean(y_prob[y_test==0])),
        "ci_95_bootstrap": {
            "mean_survived": {"lower": float(ci["survivors"][0]), "upper": float(ci["survivors"][1])},
            "mean_died": {"lower": float(ci["decedents"][0]), "upper": float(ci["decedents"][1])},
            "mean_gap": {"lower": float(ci["gap"][0]), "upper": float(ci["gap"][1])},
        }
    },
}

out_path = OUTPUT_DIR / f"{MODEL_NAME}_{COHORT_NAME}_{time_stamp}.json"

with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved JSON:", out_path)